# AIC 2026 — Keyframe extraction · TransNetV2

**Input:** `oh-i-ace/aic-videos`  
**Output:** `aqpahm/aic2026-keyframes-transnetv2`  
**Per-video artifacts:** `keyframes.tar`, `frames.parquet`, `_SUCCESS.json`

Notebook nguồn chạy độc lập trên **Google Colab hoặc Kaggle**. Bật GPU, Internet
và secret `HF_TOKEN`. Job resumable: chỉ video có marker hợp lệ mới được bỏ qua.


In [ ]:
%pip install -q -U "huggingface_hub>=0.34,<2"

import os
import sys
import tempfile
from pathlib import Path

from huggingface_hub import HfApi

def detect_runtime():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    if RUNTIME == "colab":
        from google.colab import userdata
        value = userdata.get(name)
    elif RUNTIME == "kaggle":
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
    if not value:
        raise RuntimeError(
            f"Missing {name}. Add it as a Colab/Kaggle secret and enable notebook access."
        )
    return value


def hosted_work_root(job_name):
    if RUNTIME == "colab":
        return Path("/content") / job_name
    if RUNTIME == "kaggle":
        return Path("/kaggle/temp") / job_name
    return Path(tempfile.gettempdir()) / job_name

RUNTIME = detect_runtime()
HF_TOKEN = read_secret("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
account = api.whoami()
print("Runtime:", RUNTIME)
print("Hugging Face:", account["name"])


In [ ]:
OUTPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"

api.create_repo(
    repo_id=OUTPUT_REPO,
    repo_type="dataset",
    private=False,
    exist_ok=True,
)

print("Dataset output:")
print(f"https://huggingface.co/datasets/{OUTPUT_REPO}")

In [ ]:
import torch

assert torch.cuda.is_available(), "Chưa bật GPU trong Settings"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip install -q transnetv2-pytorch imagehash open_clip_torch pyarrow opencv-python-headless

In [ ]:
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from transnetv2_pytorch import TransNetV2

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in the Colab/Kaggle runtime before running this job")

GPU_IDS = list(range(torch.cuda.device_count()))
DEVICES = [f"cuda:{device_id}" for device_id in GPU_IDS]
WORK_DIR = hosted_work_root("aic-keyframes")
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRANSNET_WEIGHTS_REPO = "ByteDance/shot2story"
TRANSNET_WEIGHTS_REVISION = api.model_info(TRANSNET_WEIGHTS_REPO, token=HF_TOKEN).sha
weights_path = hf_hub_download(
    repo_id=TRANSNET_WEIGHTS_REPO,
    filename="transnetv2-pytorch-weights.pth",
    revision=TRANSNET_WEIGHTS_REVISION,
    token=HF_TOKEN,
)

transnet_models = []
for device in DEVICES:
    detector = TransNetV2(device=device)
    detector.load_state_dict(
        torch.load(weights_path, map_location=device, weights_only=True)
    )
    detector.eval()
    transnet_models.append(detector)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
print("Runtime:", RUNTIME)
print("GPU workers:", len(GPU_IDS))
for device_id in GPU_IDS:
    properties = torch.cuda.get_device_properties(device_id)
    print(f"  cuda:{device_id}: {properties.name}, {properties.total_memory / 1024**3:.1f} GiB")


In [ ]:
SOURCE_REPO = "oh-i-ace/aic-videos"
INPUT_REVISION = api.dataset_info(SOURCE_REPO, token=HF_TOKEN).sha

source_files = api.list_repo_files(
    repo_id=SOURCE_REPO,
    repo_type="dataset",
    revision=INPUT_REVISION,
    token=HF_TOKEN,
)

archives = sorted(
    name for name in source_files
    if name.lower().endswith(".zip")
    and name.startswith("Videos_")
)

print(f"Tìm thấy {len(archives)} file video:")
for name in archives:
    print(" -", name)

In [ ]:
ARCHIVE_NAMES = [
    "Videos_L21_a.zip",
    "Videos_L22_a.zip",
    "Videos_L23_a.zip",
    "Videos_L24_a.zip",
    "Videos_L25_a.zip",
    "Videos_L26_a.zip",
    "Videos_L26_b.zip",
    "Videos_L26_c.zip",
    "Videos_L26_d.zip",
    "Videos_L26_e.zip",
    "Videos_L27_a.zip",
    "Videos_L28_a.zip",
    "Videos_L29_a.zip",
    "Videos_L30_a.zip",
]

missing_archives = [name for name in ARCHIVE_NAMES if name not in archives]
assert not missing_archives, f"Không tìm thấy trên source repo: {missing_archives}"

print(f"Sẽ xử lý {len(ARCHIVE_NAMES)} archive:")
for name in ARCHIVE_NAMES:
    print(" -", name)

In [ ]:
import open_clip

clip_models = []
clip_preprocess = None
for device in DEVICES:
    current_model, _, current_preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32",
        pretrained="laion2b_s34b_b79k",
        device=device,
    )
    current_model.eval()
    for parameter in current_model.parameters():
        parameter.requires_grad_(False)
    clip_models.append(current_model)
    if clip_preprocess is None:
        clip_preprocess = current_preprocess

print(f"Loaded {len(clip_models)} OpenCLIP replica(s)")


In [ ]:
import csv
import shutil
import imagehash
import cv2
import numpy as np
import pandas as pd

from PIL import Image
from pathlib import Path


CUT_THRESHOLD = 0.50
GAP_SECONDS = 2.0
SHORT_SHOT_SECONDS = 4.0
PHASH_DISTANCE = 6
COSINE_THRESHOLD = 0.965
JPEG_QUALITY = 92

PREVIEW_DIR = WORK_DIR / "preview"


def select_candidates(predictions, fps, detector):
    """Lấy ứng viên keyframe theo từng shot."""
    predictions = np.asarray(predictions).reshape(-1)

    scenes = detector.predictions_to_scenes(
        predictions,
        threshold=CUT_THRESHOLD,
    )

    candidates = []

    for shot_id, (start, end) in enumerate(scenes):
        start, end = int(start), int(end)
        duration = (end - start + 1) / fps

        if duration < 0.5:
            indexes = [(start + end) // 2]

        elif duration < SHORT_SHOT_SECONDS:
            indexes = [
                start + round(fraction * (end - start))
                for fraction in (0.15, 0.50, 0.85)
            ]

        else:
            stable_start = (
                start
                if start == 0
                else min(end, start + max(1, round(0.5 * fps)))
            )

            step = max(1, round(GAP_SECONDS * fps))
            indexes = list(range(stable_start, end + 1, step))

            if not indexes or indexes[-1] != end:
                indexes.append(end)

        for frame_idx in sorted(set(indexes)):
            candidates.append((frame_idx, shot_id))

    return sorted(set(candidates))


def encode_image(pil_image, clip_model, device):
    tensor = clip_preprocess(pil_image).unsqueeze(0).to(device, non_blocking=True)

    with torch.inference_mode():
        embedding = clip_model.encode_image(tensor)
        embedding = embedding / embedding.norm(dim=-1, keepdim=True)

    return embedding[0].float().cpu().numpy()


def extract_keyframes(video_path, video_id, predictions, clip_model, device, detector):
    cap = cv2.VideoCapture(str(video_path))

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        raise RuntimeError(f"{video_id}: FPS không hợp lệ: {fps}")

    candidates = select_candidates(predictions, fps, detector)
    candidate_map = dict(candidates)

    output_dir = PREVIEW_DIR / video_id
    output_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    last_kept = {}
    frame_idx = 0

    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx not in candidate_map:
            frame_idx += 1
            continue

        shot_id = candidate_map[frame_idx]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(rgb)

        current_hash = imagehash.phash(pil_image)
        previous = last_kept.get(shot_id)

        force_keep = (
            previous is None
            or (frame_idx - previous["frame_idx"]) / fps >= GAP_SECONDS
        )

        phash_distance = None
        cosine_similarity = None

        if previous is not None:
            phash_distance = current_hash - previous["phash"]

            if phash_distance <= PHASH_DISTANCE and not force_keep:
                frame_idx += 1
                continue

        embedding = encode_image(pil_image, clip_model, device)

        if previous is not None:
            cosine_similarity = float(
                np.dot(embedding, previous["embedding"])
            )

            if cosine_similarity >= COSINE_THRESHOLD and not force_keep:
                frame_idx += 1
                continue

        sample_n = len(rows) + 1
        filename = f"{sample_n:04d}.jpg"
        image_path = output_dir / filename

        cv2.imwrite(
            str(image_path),
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
        )

        rows.append({
            "video_id": video_id,
            "sample_n": sample_n,
            "shot_id": shot_id,
            "frame_idx": frame_idx,
            "timestamp_sec": round(frame_idx / fps, 6),
            "image_path": f"{video_id}/{filename}",
            "transnet_score": round(
                float(np.asarray(predictions).reshape(-1)[frame_idx]),
                6,
            ),
            "phash_distance": phash_distance,
            "previous_cosine_similarity": cosine_similarity,
        })

        last_kept[shot_id] = {
            "frame_idx": frame_idx,
            "phash": current_hash,
            "embedding": embedding,
        }

        frame_idx += 1

    cap.release()

    print(
        f"{video_id}: "
        f"{len(candidates)} candidates → {len(rows)} keyframes"
    )

    return rows

In [ ]:
import json
import re
import tarfile
import zipfile
import time
import threading
from huggingface_hub import CommitOperationAdd, hf_hub_download

DOWNLOAD_ROOT = WORK_DIR / "downloads"
PRODUCTION_ROOT = WORK_DIR / "production"
PACKAGE_ROOT = WORK_DIR / "packages"

for directory in (DOWNLOAD_ROOT, PRODUCTION_ROOT, PACKAGE_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

UPLOAD_LOCK = threading.Lock()
UPLOAD_BATCH_VIDEOS = 3
PENDING_OPERATIONS = []
PENDING_VIDEO_DIRS = []
PENDING_REMOTE_PATHS = []
REMOTE_FILES_LOCK = threading.Lock()
REMOTE_FILES = set()

# extract_keyframes ghi ảnh vào biến global này.
PREVIEW_DIR = PRODUCTION_ROOT


def retry(operation, attempts=7):
    """Retry các thao tác mạng; không retry lỗi xử lý video."""
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as exc:
            if attempt == attempts:
                raise
            delay = min(300, 5 * (2 ** (attempt - 1)))
            print(
                f"Thao tác mạng lỗi ({type(exc).__name__}); "
                f"thử lại sau {delay}s [{attempt}/{attempts}]"
            )
            time.sleep(delay)


def level_from_archive(archive_name):
    match = re.fullmatch(r"Videos_(L\d{2})_[a-z]\.zip", archive_name)
    if not match:
        raise ValueError(f"Tên archive không hợp lệ: {archive_name}")
    return match.group(1)


def remote_video_completed(level, video_id):
    prefix = f"data/{level}/{video_id}"
    required = ("keyframes.tar", "frames.parquet", "_SUCCESS.json")
    with REMOTE_FILES_LOCK:
        return all(f"{prefix}/{name}" in REMOTE_FILES for name in required)


def package_and_upload(level, archive_name, video_id, rows):
    video_frame_dir = PRODUCTION_ROOT / video_id
    package_dir = PACKAGE_ROOT / video_id

    shutil.rmtree(package_dir, ignore_errors=True)
    package_dir.mkdir(parents=True, exist_ok=True)

    tar_path = package_dir / "keyframes.tar"
    parquet_path = package_dir / "frames.parquet"

    frame_files = sorted(video_frame_dir.glob("*.jpg"))
    if len(frame_files) != len(rows):
        raise RuntimeError(
            f"{video_id}: {len(frame_files)} ảnh nhưng metadata có {len(rows)} dòng"
        )

    pd.DataFrame(rows).to_parquet(parquet_path, index=False)

    with tarfile.open(tar_path, mode="w") as tar:
        tar.add(video_frame_dir, arcname=video_id)

    remote_dir = f"data/{level}/{video_id}"

    # Tạo marker cục bộ rồi commit 3 artifact cùng lúc. Marker là operation cuối.
    marker = {
        "video_id": video_id,
        "level": level,
        "keyframe_count": len(rows),
        "source_repo": SOURCE_REPO,
        "source_revision": INPUT_REVISION,
        "source_archive": archive_name,
        "schema_version": 1,
        "pipeline": {
            "shot_detector": "TransNetV2",
            "weights_repo": TRANSNET_WEIGHTS_REPO,
            "weights_revision": TRANSNET_WEIGHTS_REVISION,
            "cut_threshold": CUT_THRESHOLD,
            "gap_seconds": GAP_SECONDS,
            "short_shot_seconds": SHORT_SHOT_SECONDS,
            "phash_distance": PHASH_DISTANCE,
            "embedding_model": "OpenCLIP ViT-B-32",
            "cosine_threshold": COSINE_THRESHOLD,
            "jpeg_quality": JPEG_QUALITY,
        },
        "completed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }

    marker_path = package_dir / "_SUCCESS.json"
    marker_path.write_text(
        json.dumps(marker, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    operations = [
        CommitOperationAdd(
            path_in_repo=f"{remote_dir}/keyframes.tar",
            path_or_fileobj=str(tar_path),
        ),
        CommitOperationAdd(
            path_in_repo=f"{remote_dir}/frames.parquet",
            path_or_fileobj=str(parquet_path),
        ),
        CommitOperationAdd(
            path_in_repo=f"{remote_dir}/_SUCCESS.json",
            path_or_fileobj=str(marker_path),
        ),
    ]
    with UPLOAD_LOCK:
        PENDING_OPERATIONS.extend(operations)
        PENDING_VIDEO_DIRS.append(package_dir)
        PENDING_REMOTE_PATHS.extend(
            f"{remote_dir}/{name}"
            for name in ("keyframes.tar", "frames.parquet", "_SUCCESS.json")
        )
        if len(PENDING_VIDEO_DIRS) >= UPLOAD_BATCH_VIDEOS:
            flush_pending_uploads_locked()


def flush_pending_uploads_locked():
    if not PENDING_OPERATIONS:
        return
    video_ids = [directory.name for directory in PENDING_VIDEO_DIRS]
    operations = list(PENDING_OPERATIONS)
    retry(
        lambda: api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add keyframes for {video_ids[0]} through {video_ids[-1]}",
        )
    )
    with REMOTE_FILES_LOCK:
        REMOTE_FILES.update(PENDING_REMOTE_PATHS)
    for directory in PENDING_VIDEO_DIRS:
        shutil.rmtree(directory, ignore_errors=True)
    print(f"✅ Uploaded {len(video_ids)} keyframe videos in one commit")
    PENDING_OPERATIONS.clear()
    PENDING_VIDEO_DIRS.clear()
    PENDING_REMOTE_PATHS.clear()


def flush_pending_uploads():
    with UPLOAD_LOCK:
        flush_pending_uploads_locked()


def process_archive(archive_name, transnet, clip_model, device_id):
    device = f"cuda:{device_id}"
    level = level_from_archive(archive_name)
    archive_path = None
    completed = 0
    skipped = 0
    failures = []

    try:
        total, used, free = shutil.disk_usage(WORK_DIR)
        print(f"Dung lượng trống: {free / 1024**3:.1f} GB")

        archive_path = Path(
            retry(
                lambda: hf_hub_download(
                    repo_id=SOURCE_REPO,
                    repo_type="dataset",
                    filename=archive_name,
                    revision=INPUT_REVISION,
                    token=HF_TOKEN,
                    local_dir=DOWNLOAD_ROOT,
                )
            )
        )

        with zipfile.ZipFile(archive_path) as archive:
            members = sorted(
                name for name in archive.namelist()
                if name.lower().endswith((".mp4", ".avi", ".mkv", ".mov"))
            )
            if not members:
                raise RuntimeError(f"{archive_name}: ZIP không chứa video")

            print(f"{archive_name}: {len(members)} video nguồn")

            for position, member in enumerate(members, start=1):
                video_id = Path(member).stem
                video_path = WORK_DIR / f"{video_id}{Path(member).suffix.lower()}"

                print(f"\n[{position}/{len(members)}] {video_id}")

                try:
                    if remote_video_completed(level, video_id):
                        print("Đã có _SUCCESS.json → bỏ qua")
                        skipped += 1
                        continue

                    shutil.rmtree(PRODUCTION_ROOT / video_id, ignore_errors=True)

                    with archive.open(member) as source, video_path.open("wb") as destination:
                        shutil.copyfileobj(source, destination)

                    with torch.inference_mode():
                        _, predictions, _ = transnet.predict_video(str(video_path))
                    rows = extract_keyframes(
                        video_path=video_path,
                        video_id=video_id,
                        predictions=predictions,
                        clip_model=clip_model,
                        device=device,
                        detector=transnet,
                    )

                    if not rows:
                        raise RuntimeError("không tạo được keyframe")

                    package_and_upload(level, archive_name, video_id, rows)
                    completed += 1
                    print(f"Đã tạo {len(rows)} keyframe; đang chờ/đã upload theo batch")

                except Exception as exc:
                    failures.append({
                        "archive": archive_name,
                        "video_id": video_id,
                        "error": f"{type(exc).__name__}: {exc}",
                    })
                    print(f"❌ {video_id}: {type(exc).__name__}: {exc}")

                finally:
                    if video_path.exists():
                        video_path.unlink()
                    shutil.rmtree(PRODUCTION_ROOT / video_id, ignore_errors=True)
                    shutil.rmtree(PACKAGE_ROOT / video_id, ignore_errors=True)
                    with torch.cuda.device(device_id):
                        torch.cuda.empty_cache()

    finally:
        if archive_path is not None and archive_path.exists():
            archive_path.unlink()

    print(f"\nHoàn thành archive: {archive_name}")
    print("Video mới xử lý:", completed)
    print("Video đã bỏ qua:", skipped)
    print("Video lỗi:", len(failures))

    return {
        "archive": archive_name,
        "completed": completed,
        "skipped": skipped,
        "failures": failures,
    }

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from queue import Queue

REMOTE_FILES.update(
    retry(lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"))
)
print("Remote artifact paths:", len(REMOTE_FILES))

run_results = []
archive_failures = []
device_pool = Queue()
for model_index in range(len(GPU_IDS)):
    device_pool.put(model_index)


def process_archive_on_available_gpu(archive_name):
    model_index = device_pool.get()
    try:
        return process_archive(
            archive_name,
            transnet_models[model_index],
            clip_models[model_index],
            GPU_IDS[model_index],
        )
    finally:
        device_pool.put(model_index)


with ThreadPoolExecutor(max_workers=len(GPU_IDS)) as executor:
    futures = {
        executor.submit(process_archive_on_available_gpu, archive_name): archive_name
        for archive_name in ARCHIVE_NAMES
    }
    for future in as_completed(futures):
        archive_name = futures[future]
        try:
            run_results.append(future.result())
        except Exception as error:
            archive_failures.append({
                "archive": archive_name,
                "error": f"{type(error).__name__}: {error}",
            })
            print(f"❌ Archive {archive_name}: {type(error).__name__}: {error}")

flush_pending_uploads()

video_failures = [
    failure
    for result in run_results
    for failure in result["failures"]
]

print("=" * 72)
print("TỔNG KẾT L21 → L30")
print("GPU workers:", len(GPU_IDS))
print("Archive hoàn tất vòng chạy:", len(run_results), "/", len(ARCHIVE_NAMES))
print("Video mới xử lý:", sum(x["completed"] for x in run_results))
print("Video đã bỏ qua:", sum(x["skipped"] for x in run_results))
print("Video lỗi:", len(video_failures))
print("Archive không mở/xử lý được:", len(archive_failures))

if video_failures:
    print("Các video cần chạy lại:")
    for item in video_failures:
        print(f" - {item['archive']} / {item['video_id']}: {item['error']}")
if archive_failures:
    print("Các archive cần chạy lại:")
    for item in archive_failures:
        print(f" - {item['archive']}: {item['error']}")
if video_failures or archive_failures:
    raise RuntimeError(
        "Còn mục lỗi. Run All lại; video có đủ 3 artifact sẽ tự được bỏ qua."
    )
print("✅ ĐÃ XỬ LÝ XONG TOÀN BỘ VIDEO BTC CUNG CẤP")
